In [ ]:
from arize.experimental.datasets import ArizeDatasetsClient
import os
from dotenv import load_dotenv

load_dotenv()

HOST = os.getenv("ARIZE_GRPC_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
DATASET_ID = "RGF0YXNldDoxNToxZnFp" # taken from 02_dataset.ipynb

client = ArizeDatasetsClient(
    api_key=API_KEY,
    host=HOST
)

In [ ]:
df = client.get_dataset(space_id=SPACE_ID, dataset_id=DATASET_ID)
df

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
chain = llm | StrOutputParser()

def sample_task(dataset_row) -> str:
  return chain.invoke(dataset_row.get("question"))

In [ ]:
sample_task(df.iloc[0])

In [ ]:
from typing import Any, Optional, Mapping
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    LLMEvaluator,
    JSONSerializable,
    TaskOutput
)
from openai import OpenAI

openai_client = OpenAI()

eval_prompt_template = """
In this task, you will be presented with a query, some context and a response. The response
is generated to the question based on the context. The response may contain false
information. You must use the context to determine if the response to the question
contains false information, if the response is a hallucination of facts. Your objective is
to determine whether the response text contains factual information and is not a
hallucination. A 'hallucination' refers to a response that is not based on the context or
assumes information that is not available in the context. Your response should be a single
word: either 'factual' or 'hallucinated', and it should not include any other text or
characters. 'hallucinated' indicates that the response provides factually inaccurate
information to the query based on the context. 'factual' indicates that the response to
the question is correct relative to the context, and does not contain made up
information. Please read the query and context carefully before determining your
response.

[BEGIN DATA]
************
[Query]: {input}
************
[Response]: {output}
************
[END DATA]

Is the response above factual or hallucinated based on the query and context?
"""

class SampleEvaluator(LLMEvaluator):
    def evaluate(
        self,
        *,
        dataset_row: Optional[Mapping[str, JSONSerializable]] = None,
        output: Optional[TaskOutput] = None,
        **_: Any,
    ) -> EvaluationResult:
        message_content = eval_prompt_template.format(
            input=dataset_row["question"], output=output
        )
        
        response = openai_client.chat.completions.create(
            model="gpt-4o", messages=[{"role": "user", "content": message_content}]
        )
        response_message_content = response.choices[0].message.content.lower().strip()
        
        is_good = response_message_content == "factual"

        return EvaluationResult(
            score=float(is_good),
            label=response_message_content,
            explanation=response_message_content
        )

In [ ]:
results_df = client.run_experiment(
    space_id=SPACE_ID, 
    dataset_id=DATASET_ID,
    task=sample_task, 
    evaluators=[SampleEvaluator()], #include your evaluation functions here 
    experiment_name="llm-eval",
    concurrency=10,
    dry_run=False
)